# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*I chose a line chart because it shows clearly how population has grown over time across the six continents. I used percentage growth relative to 1970 to make sure that the comparisons were fair, since different continents started out with different population sizes. And I also choose different colours so its easier to tell the continent apart. Also named the fastest growing population in the title so the possibility of confusion is less likely*

In [4]:
# Q1
# YOUR CODE HERE


# Q1

# Calculate total population for each continent and year
continent_pop = df_long.groupby(['Continent', 'Year'])['Population'].sum().reset_index()

# Get the population of each continent in 1970
baseline = continent_pop[continent_pop['Year'] == 1970][['Continent', 'Population']]
baseline = baseline.rename(columns={'Population': 'Population_1970'})

# Calculate percentage growth relative to 1970
growth = continent_pop.merge(baseline, on='Continent')
growth['Growth (%)'] = (growth['Population'] / growth['Population_1970'] - 1) * 100

# Create line chart
fig = px.line(
    growth,
    x='Year',
    y='Growth (%)',
    color='Continent',
    markers=True,
    title="Africa experienced the fastest population growth since 1970",
    color_discrete_sequence=[
    'black',
    'red',
    'lightblue',
    'yellow',
    'green',
    'orange'
],
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Population growth (%)',
    legend_title=None
)

fig.update_yaxes(showgrid=False)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*I choose a horizontal bar chart  because it allows us to compare the increase in population of the 15 countries clearly and identify which countries added the most people. The horizontal layout makes country names easier to read. I used Red to highlight African countries, while black represents all other countries so the questions of which countried are African can be easily asertained from the chart.


In [12]:
# Q2
# YOUR CODE HERE
top_15['Colour'] = top_15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other countries'
)

In [13]:
fig = px.bar(
    top_15.sort_values('Increase'),
    x='Increase',
    y='Country',
    orientation='h',
    color='Colour',
    color_discrete_map={
        'Africa': 'red',
        'Other countries': 'black'
    },
    title='Top 15 Countries with the Largest Population Increase (1970–2022)',
    labels={
        'Increase': 'Population Increase',
        'Country/Territory': 'Country'
    },
    template='plotly_white'
)

fig.update_layout(
    yaxis_title='',
    xaxis_title='Additional People',
    legend_title='Country Group',
    xaxis_showgrid=False
)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I think a scatter plot is a good choice because it helps us see if there is a relationship between a specific country's land area and its population. Each dot represents a country, while different colours make it easy to compare the continents.

In [18]:
# Q3
# YOUR CODE HERE
fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    color_discrete_sequence=[
    'black',
    'blue',
    'red',
    'yellow',
    'green',
    'orange'
],
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Relationship Between Country Land Area and Population (2022)',
    labels={
        'Area (km²)': 'Land Area (km²)',
        '2022 Population': 'Population (2022)'
    },
    template='plotly_white'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A box plot was the best choice because it shows how population density changes between countries within a continent, rather than only showing the average. Different colours help separate the continents, while the scale makes countries with very different population densities easier to compare.

In [20]:
# Q4
# YOUR CODE HERE
df['Population Density'] = df['2022 Population'] / df['Area']

In [21]:
fig = px.box(
    df,
    x='Continent',
    y='Population Density',
    color='Continent',
    color_discrete_sequence=[
        'black', 'blue', 'red',
        'yellow', 'green', 'orange'
    ],
    points='outliers',
    log_y=True,
    title='Population Density Across Continents (2022)',
    labels={
        'Continent': 'Continent',
        'Population Density': 'People per km²'
    },
    template='plotly_white'
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A horizontal bar chart waws a good choice because it makes it easy to compare population changes across countries. Countries with positive values show population growth, while negative values show population decline. The different colours represent the continents, helping us see where these changes are happening around the world. Sorting the bars makes the differences in continents easier to spot, while the zero line clearly separates countries that grew from those that shrank.

In [22]:
# Q5
# YOUR CODE HERE
df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population']
) * 100

In [25]:
fig = px.bar(
    df.sort_values('Population Change (%)'),
    x='Population Change (%)',
    y='Country',
    orientation='h',
    color='Continent',
    color_discrete_sequence=[
        'black', 'blue', 'red',
        'yellow', 'green', 'orange'
    ],
    hover_name='Country',
    title='Percentage Population Change by Country (2000–2022)',
    labels={
        'Population Change (%)': 'Population Change (%)',
        'Country/Territory': 'Country'
    },
    template='plotly_white',
    height=1200
)

fig.add_vline(x=0, line_dash='dash', line_color='grey')
fig.update_layout(
    xaxis_title='Percentage Change (%)',
    yaxis_title='',
    showlegend=True
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A treemap is the best choice because it shows the continents and their countries together in one figure. The size of each rectangle represents the country's population, making it easy to compare their contributions to the world population. Grouping countries by continent makes the overall vizualisation easier to understand, while the different colours help separate the continents.

In [27]:
# Q6
# YOUR CODE HERE

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    title='Distribution of World Population by Continent and Country (2022)',
    color='Continent',
    color_discrete_sequence=[
        'black', 'blue', 'red',
        'yellow', 'green', 'orange'
    ],
    template='plotly_white'
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A stacked bar chart is a good choice because it allows us to compare how the world's population has been divided between the four groups over time. Each bar represents a year and adds up to 100%, making it easy to see how the population shares of Africa, Asia, Europe and the rest of the world have changed.

In [28]:
# Q7
# YOUR CODE HERE

population = df_long.groupby(
    ['Continent', 'Year']
)['Population'].sum().reset_index()

population['Group'] = population['Continent'].apply(
    lambda x: x if x in ['Africa', 'Asia', 'Europe']
    else 'Rest of the world'
)

population = population.groupby(
    ['Group', 'Year']
)['Population'].sum().reset_index()

world_total = population.groupby('Year')['Population'].transform('sum')

population['Share (%)'] = (
    population['Population'] / world_total
) * 100

In [30]:
fig = px.bar(
    population,
    x='Year',
    y='Share (%)',
    color='Group',
    barmode='stack',
    category_orders={
        'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']
    },
    color_discrete_map={
        'Africa': 'red',
        'Asia': 'blue',
        'Europe': 'black',
        'Rest of the world': 'orange'
    },
    title='World Population Share by Continent (1970–2022)',
    labels={
        'Year': 'Year',
        'Share (%)': 'Share of World Population (%)'
    },
    template='plotly_white'
)

fig.update_layout(
    yaxis_range=[0, 100],
    yaxis_title='Share of World Population (%)'
)

fig.show()